# ST-OMR Meter V5-2R — TRAIN-only Class / Margin / Gradient Audit

Single-run, fail-closed evidence harness pinned to exact CI-green implementation commit `85c0b0083792e8b9ec60ee632cfc7015e885d548`. This notebook does not train, call autograd/backward, take optimizer steps, tune thresholds/bias, mutate checkpoints, or open Historical VALIDATION, First-30, V5 VAL, or FINAL_HOLDOUT.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import time

EXPECTED_HEAD = "85c0b0083792e8b9ec60ee632cfc7015e885d548"
EXPECTED_CI_RUN_ID = 32660114964
REPOSITORY = "khfy7wpr5p-maker/st-omr-training"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
REPO = Path("/content/st-omr-training")
MYDRIVE = Path("/content/drive/MyDrive")

# Mount Drive and bind only the already-approved TRAIN/checkpoint surfaces.
if not MYDRIVE.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = MYDRIVE / "TEST" / "METER_V2_1500_PACKAGE_AB_CLEAN"
CHECKPOINT_ROOT = MYDRIVE / "ST-OMR-METER-SPECIALISTS"
M4A_ROOT = CHECKPOINT_ROOT / "m4a-234-digit-specialist-dataset-freeze-v2"
D10_ROOT = (
    MYDRIVE
    / "ST-OMR-D10"
    / "stage7d10-authoritative-562c8fcfabf1b41573f1ef591d88ae65335ce16a"
)
for name, path in {
    "DATA_ROOT": DATA_ROOT,
    "CHECKPOINT_ROOT": CHECKPOINT_ROOT,
    "M4A_ROOT": M4A_ROOT,
    "D10_ROOT": D10_ROOT,
}.items():
    if not path.is_dir():
        raise RuntimeError(f"{name} bulunamadi: {path}")
print("DRIVE CHECK = PASS")

# Fetch the immutable implementation commit directly; branch movement is irrelevant.
if not REPO.exists():
    subprocess.check_call(["git", "clone", "--no-checkout", REPO_URL, str(REPO)])
elif not (REPO / ".git").is_dir():
    raise RuntimeError(f"REPO git repository degil: {REPO}")

remotes = subprocess.check_output(
    ["git", "-C", str(REPO), "remote"], text=True
).split()
if "origin" not in remotes:
    subprocess.check_call(["git", "-C", str(REPO), "remote", "add", "origin", REPO_URL])
else:
    subprocess.check_call(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL])
subprocess.check_call(
    ["git", "-C", str(REPO), "fetch", "origin", EXPECTED_HEAD, "--depth", "1"]
)
fetched_head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if fetched_head != EXPECTED_HEAD:
    raise RuntimeError(f"FETCH_HEAD mismatch: expected={EXPECTED_HEAD} actual={fetched_head}")
subprocess.check_call(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD])
actual_head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
if actual_head != EXPECTED_HEAD:
    raise RuntimeError(f"HEAD mismatch: expected={EXPECTED_HEAD} actual={actual_head}")
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Repository worktree temiz degil")
print("REPOSITORY CHECK = PASS")
print("HEAD =", actual_head)
print("CI RUN ID =", EXPECTED_CI_RUN_ID)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from st_omr_training import meter_v5_1_bbox_pilot as v51
from st_omr_training import meter_v5_2b_specialist_adaptation as v52b
from st_omr_training import meter_v5_2r_train_class_margin_gradient_audit_v1 as audit
print("MODULE IMPORT = PASS")

DIGIT2_FROZEN = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT_ROOT, v52b.DIGIT2_SHA256)
DIGIT3_FROZEN = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT_ROOT, v52b.DIGIT3_SHA256)
print("FROZEN CHECKPOINTS = PASS")
print("2-AI =", DIGIT2_FROZEN)
print("3-AI =", DIGIT3_FROZEN)

ANN_DIR = DATA_ROOT / "annotations"
REPORT_PATH = ANN_DIR / audit.REPORT_NAME
ENVELOPE_PATH = ANN_DIR / f"v5_2r_execution_envelope_{EXPECTED_HEAD}.json"
for path in (REPORT_PATH, ENVELOPE_PATH):
    if path.exists():
        raise RuntimeError(f"Refusing overwrite/rerun: {path}")
print("OUTPUT GUARD = PASS")

required_safety = {
    "training": False,
    "autograd_grad_used": False,
    "backward": False,
    "optimizer_steps": 0,
    "checkpoint_write": False,
    "candidate_checkpoint_mutation": False,
    "objective_changed": False,
    "new_objective_selected": False,
    "threshold_tuning": False,
    "bias_tuning": False,
    "historical_validation_opened": False,
    "historical_validation_retention_report_read": False,
    "first30_opened": False,
    "v5_validation_opened": False,
    "final_holdout_locked": True,
    "digit4_frozen": True,
    "repair_selected": False,
    "repair_training_authorized": False,
    "production_promotion": False,
}
boundary = audit.safety_boundary()
for key, expected in required_safety.items():
    if boundary.get(key) != expected:
        raise RuntimeError(f"Safety boundary mismatch: {key}={boundary.get(key)!r}")
print("SAFETY BOUNDARY = PASS")
print("TRAINING=False | AUTOGRAD=False | BACKWARD=False | OPTIMIZER_STEPS=0")
print("HISTORICAL_VALIDATION=CLOSED | FIRST-30=CLOSED | V5_VAL=CLOSED")
print("FINAL_HOLDOUT=LOCKED | 4-AI=FROZEN")

started = time.time()
def progress(processed, total, phase):
    if processed == 1 or processed == total or processed % 2048 == 0:
        print(phase, f"{processed}/{total}", f"| elapsed={int(time.time() - started)}s")

report = audit.run_train_class_margin_gradient_audit_v1(
    DATA_ROOT,
    m4a_root=M4A_ROOT,
    d10_root=D10_ROOT,
    digit2_frozen=DIGIT2_FROZEN,
    digit3_frozen=DIGIT3_FROZEN,
    progress=progress,
)

# Bind the produced report to the immutable repository commit without mutating it.
if not REPORT_PATH.is_file():
    raise RuntimeError(f"Audit report yazilmadi: {REPORT_PATH}")
report_bytes = REPORT_PATH.read_bytes()
saved_report = json.loads(report_bytes.decode("utf-8"))
if saved_report != report:
    raise RuntimeError("In-memory ve saved V5-2R report farkli")
for key, expected in required_safety.items():
    if report.get(key) != expected:
        raise RuntimeError(f"Saved report safety mismatch: {key}={report.get(key)!r}")
post_run_head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
if post_run_head != EXPECTED_HEAD:
    raise RuntimeError(f"Post-run HEAD mismatch: {post_run_head}")
report_sha256 = hashlib.sha256(report_bytes).hexdigest()
envelope = {
    "schema": "st-omr-meter-v5-2r-exact-sha-execution-envelope-v1",
    "repository": REPOSITORY,
    "expected_head": EXPECTED_HEAD,
    "actual_head_before_run": actual_head,
    "actual_head_after_run": post_run_head,
    "ci_run_id": EXPECTED_CI_RUN_ID,
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "audit_report_name": audit.REPORT_NAME,
    "audit_report_sha256": report_sha256,
    "audit_schema": report.get("schema"),
    "candidate_checkpoint_sha256": report.get("candidate_checkpoint_sha256"),
    "frozen_checkpoint_sha256": report.get("frozen_checkpoint_sha256"),
    "safety_boundary": {key: report[key] for key in required_safety},
}
v51._atomic_write_json(ENVELOPE_PATH, envelope)
envelope_sha256 = hashlib.sha256(ENVELOPE_PATH.read_bytes()).hexdigest()

print()
print("============================================")
print("V5-2R EXACT-SHA EXECUTION RESULT")
print("============================================")
for digit in ("2", "3"):
    item = report["per_specialist"][digit]
    print()
    print(f"========== {digit}-AI ==========")
    print("CLASS BALANCE =", item["class_balance"])
    print("HISTORICAL POSITIVE TRANSITIONS =", item["historical_positive_transition_matrix"])
    print("HEAD GEOMETRY =", item["head_geometry"])
    print("FROZEN OBJECTIVE =", item["gradient_and_bce_at_frozen_head"]["objective_reconstruction"])
    print("CANDIDATE OBJECTIVE =", item["gradient_and_bce_at_candidate_head"]["objective_reconstruction"])
    print("FROZEN GRADIENT COSINES =", item["gradient_and_bce_at_frozen_head"]["gradient_conflict_cosine_matrix"])
    print("CANDIDATE GRADIENT COSINES =", item["gradient_and_bce_at_candidate_head"]["gradient_conflict_cosine_matrix"])
print()
print("EXACT SHA EXECUTION = PASS")
print("HEAD =", post_run_head)
print("REPORT =", REPORT_PATH)
print("REPORT SHA256 =", report_sha256)
print("EXECUTION ENVELOPE =", ENVELOPE_PATH)
print("ENVELOPE SHA256 =", envelope_sha256)
print("TRAINING EXECUTED = False")
print("NEW OBJECTIVE SELECTED = False")
print("REPAIR TRAINING AUTHORIZED = False")
print("HISTORICAL VALIDATION = CLOSED")
print("FIRST-30 = CLOSED | V5 VAL = CLOSED | FINAL_HOLDOUT = LOCKED")
